In [45]:
import pandas as pd
import json
# Load both parquet files
new = pd.read_parquet(r"D:\LiteratureReviewCVinWC\review_output\cv4animals.parquet")
old = pd.read_parquet(r"D:\LiteratureReviewCVinWC\review_output\manual_old.parquet")


In [46]:
def reset_list_column(dataframe: pd.DataFrame, column: str) -> None:
    dataframe[column] = dataframe[column].apply(lambda _: [])

In [47]:
for c in new.columns:
    if "unverified" in c:
        reset_list_column(new, c)

In [48]:
def avg_list_length(series):
    if series.apply(lambda x: isinstance(x, list)).all():
        return series.apply(len).mean()
    return None  # or 0, depending on preference

In [49]:
def merge_columns(dataframe: pd.DataFrame, column1: str, column2: str, target_column: str = None) -> pd.DataFrame:
    """
    Merge two list columns in a pandas DataFrame by extending the list in column1
    with the values from column2, and store the result in target_column.

    Parameters:
    ----------
    dataframe : pd.DataFrame
        The input dataframe.
    column1 : str
        The name of the first list column.
    column2 : str
        The name of the second list column.
    target_column : str
        The name of the resulting merged column.

    Returns:
    -------
    pd.DataFrame
        DataFrame with the new merged column.
    """
    if target_column is None:
        target_column = column2
    dataframe[target_column] = dataframe.apply(
        lambda row: list(set((row[column1] or []) + (row[column2] or []))), axis=1
    )
    return dataframe

In [50]:
# merge Species (Text)(translated) - verified and Species (Images)(translated) - verified
new = merge_columns(new, "Species (Text)(translated) - verified", "Species (Images)(translated) - verified")
# reset Species (Images)(translated) - verified
reset_list_column(new, "Species (Images)(translated) - verified")
# Imaging Method - verified to Imaging Method (Text) - verified
new = merge_columns(new, "Imaging Method - verified", "Imaging Method (Text) - verified")
# reset Imaging Method - verified
reset_list_column(new, "Imaging Method - verified")
# merge Light Spectra (Text) - verified and Light Spectra (Images) - verified
new = merge_columns(new, "Light Spectra (Text) - verified", "Light Spectra (Images) - verified")
# reset Light Spectra (Images) - verified
reset_list_column(new, "Light Spectra (Images) - verified")
# reset HabitatVerification values
reset_list_column(new, "HabitatVerification values")

In [51]:
# Apply to all columns
avg_lengths = {col: avg_list_length(old[col]) for col in old.columns}

print(json.dumps(avg_lengths, indent=2))

{
  "file": null,
  "doi": null,
  "year": null,
  "Species (Text)(translated) - verified": 6.2438095238095235,
  "Species (Text)(translated) - unverified": 0.0,
  "Species (Images)(translated) - verified": 0.0,
  "Species (Images)(translated) - unverified": 0.0,
  "Country - verified": 1.1828571428571428,
  "Country - unverified": 0.0,
  "Imaging Method - verified": 0.0,
  "Imaging Method - unverified": 0.0,
  "Light Spectra (Text) - verified": 1.0,
  "Light Spectra (Text) - unverified": 0.0,
  "Light Spectra (Images) - verified": 0.0,
  "Light Spectra (Images) - unverified": 0.0,
  "Imaging Method (Text) - verified": 1.0057142857142858,
  "Imaging Method (Text) - unverified": 0.0,
  "CV Tasks - verified": 1.2247619047619047,
  "CV Tasks - unverified": 0.0,
  "CV Algorithms - verified": 1.859047619047619,
  "CV Algorithms - unverified": 0.0,
  "Dataset - verified": 1.2933333333333332,
  "Dataset - unverified": 0.0,
  "HabitatVerification values": 0.0,
  "ParentHabitat values": 1.76952

In [52]:
# Apply to all columns
avg_lengths = {col: avg_list_length(new[col]) for col in new.columns}

print(json.dumps(avg_lengths, indent=2))

{
  "file": null,
  "doi": null,
  "year": null,
  "Species (Text)(translated) - verified": 4.911111111111111,
  "Species (Text)(translated) - unverified": 0.0,
  "Species (Images)(translated) - verified": 0.0,
  "Species (Images)(translated) - unverified": 0.0,
  "Country - verified": 0.9111111111111111,
  "Country - unverified": 0.0,
  "Imaging Method - verified": 0.0,
  "Imaging Method - unverified": 0.0,
  "Light Spectra (Text) - verified": 1.1555555555555554,
  "Light Spectra (Text) - unverified": 0.0,
  "Light Spectra (Images) - verified": 0.0,
  "Light Spectra (Images) - unverified": 0.0,
  "Imaging Method (Text) - verified": 0.8222222222222222,
  "Imaging Method (Text) - unverified": 0.0,
  "CV Tasks - verified": 2.8666666666666667,
  "CV Tasks - unverified": 0.0,
  "CV Algorithms - verified": 7.466666666666667,
  "CV Algorithms - unverified": 0.0,
  "Dataset - verified": 4.0,
  "Dataset - unverified": 0.0,
  "HabitatVerification values": 0.0,
  "ParentHabitat values": 3.422222

In [53]:
# Concatenate rows
df_merged = pd.concat([old, new], ignore_index=True)

# Save to a new parquet file
df_merged.to_parquet(r"D:\LiteratureReviewCVinWC\review_output\manual.parquet", index=False)

In [56]:
df = df_merged
mask = df["Imaging Method (Text) - verified"].apply(lambda lst: "Acoustic Camera" in lst) & df["ParentHabitat values"].apply(lambda lst: any([x.startswith("Marine") for x in lst]))
result = df[mask]

print(result["doi"])

437    https://doi.org/10.1371/journal.pone.0267759
438    https://doi.org/10.1371/journal.pone.0273588
Name: doi, dtype: object
